# Clase 4 — Regularización y modelos lineales generalizados

**Maestría en Bioinformática y Biología de Sistemas — UNNOBA**
**Reconocimiento de Patrones en Bioinformática**

---

## Objetivos de esta clase

En la **Clase 2** vimos a Lasso brevemente como herramienta de selección. En la **Clase 3** vimos cómo PCA "achica" las direcciones de baja varianza para reducir dimensionalidad. Hoy unimos las dos ideas en un marco más general: la **regularización**, que es la forma estándar de tratar con alta dimensionalidad cuando querés hacer **predicción** (no solo exploración).

La regularización es probablemente la herramienta más importante del curso. Aparece en:
- Lasso, Ridge, Elastic Net (hoy)
- SVM (Clase 5, el parámetro C es una forma de regularización L2)
- Redes neuronales (Clase 6, weight decay, dropout)
- Hasta los árboles tienen versiones regularizadas (XGBoost)

La idea de esta clase es que terminemos con los siguientes conceptos:

1. Entender **por qué la regresión clásica falla** cuando $p > n$ y por qué se vuelve inestable con multicolinealidad.
2. Explicar **Ridge** (penalización L2): cuándo conviene, por qué siempre tiene solución, qué hace con coeficientes.
3. Explicar **Lasso** (penalización L1): por qué selecciona variables y cuáles son sus limitaciones.
4. Aplicar **Elastic Net** y entender cuándo combinar L1+L2.
5. Aplicar **regresión logística regularizada** para clasificación binaria.
6. Elegir el método correcto según las características del problema biológico.


In [ ]:
# !pip install numpy pandas matplotlib seaborn scikit-learn scipy statsmodels --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("Setup listo.")

Setup listo.


---

# Por qué falla la regresión clásica en p ≫ n

## El recordatorio de la regresión lineal clásica

La regresión lineal busca encontrar coeficientes $\beta$ que minimicen el **error cuadrático**:

$$\hat{\beta} = \arg\min_{\beta} \|y - X\beta\|^2 = \arg\min_{\beta} \sum_{i=1}^n (y_i - x_i^T \beta)^2$$

La solución analítica famosa es:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

Esto funciona perfecto cuando $n$ es grande, $p$ es chico, y las variables son razonablemente independientes. Pero la realidad bioinformática rara vez cumple esas tres condiciones.

## Problema 1: cuando p > n, no hay solución única

Si $X$ es $n \times p$ con $p > n$, la matriz $X^T X$ es de tamaño $p \times p$ pero su **rango es como mucho $n$**. Entonces $X^T X$ no es invertible (es **singular**).

¿Qué significa eso geométricamente? Que **existen infinitos $\beta$ que dan exactamente el mismo error de entrenamiento cero**. El modelo puede ajustar perfecto los $n$ puntos de training, pero hay infinitas formas de hacerlo, y todas son igualmente válidas matemáticamente.

¿Cuál de todas elegir? La regresión clásica no tiene respuesta. Cuando sklearn (o R, o cualquier software) intenta ajustar `LinearRegression` con $p > n$, internamente usa **pseudo-inversas** o **regularización numérica implícita**, pero el resultado es inestable y poco confiable.

**Caso bioinformático típico:** 50 pacientes, 20.000 genes. Cualquier subconjunto de 50 genes puede ajustar perfectamente los 50 pacientes. La "firma génica" que reportes va a ser **arbitraria**.

## Problema 2: aún con p < n, multicolinealidad rompe todo

Supongamos que tenés más muestras que variables ($n > p$). ¿La regresión clásica funciona? **Depende**.

Si dos variables están altamente correlacionadas (ejemplo: "mean radius" y "mean perimeter" de un tumor — son geométricamente casi proporcionales), el modelo no sabe cómo repartir el coeficiente entre ellas. Matemáticamente, $X^T X$ no es singular pero está **mal condicionada**: su determinante es muy chico, y la inversa se vuelve numéricamente inestable.

**Qué pasa en la práctica:**
- Los coeficientes pueden tomar valores enormes y de signo opuesto (ejemplo: $\beta_{radius} = +5000$ y $\beta_{perimeter} = -4998$).
- Pequeñas variaciones en los datos cambian drásticamente los coeficientes.
- Los intervalos de confianza son anchísimos.
- La interpretación biológica es imposible ("¿el radio aumenta o disminuye la probabilidad de cáncer?").

Esto se llama **inestabilidad por multicolinealidad** y es **muy común** en bioinformática, donde grupos de variables (genes co-regulados, features geométricas, SNPs en LD) están naturalmente correlacionadas.

## La idea de la regularización

La solución: **modificar la función objetivo agregando una penalización** que castigue coeficientes grandes:

$$\hat{\beta} = \arg\min_{\beta} \underbrace{\|y - X\beta\|^2}_{\text{ajuste a los datos}} + \lambda \cdot \underbrace{\text{penalización}(\beta)}_{\text{castigo a coef. grandes}}$$

Tres elecciones principales para la penalización:

| Nombre | Penalización | Fórmula |
|---|---|---|
| **Ridge** | L2 | $\sum_j \beta_j^2$ |
| **Lasso** | L1 | $\sum_j |\beta_j|$ |
| **Elastic Net** | L1 + L2 | $\rho \sum |\beta_j| + (1-\rho)\sum \beta_j^2$ |

El parámetro $\lambda$ (a veces llamado $\alpha$) controla **cuánto pesa la regularización**:
- $\lambda = 0$: regresión clásica sin penalización.
- $\lambda \to \infty$: todos los coeficientes son cero (modelo constante).
- $\lambda$ óptimo: en algún valor intermedio.

**Por qué esto resuelve los problemas:**
- El término penalizador **prefiere coeficientes chicos y estables**.
- Con multicolinealidad, ya no es indiferente repartir el coeficiente como sea: la penalización elige la repartición más "económica".
- Cuando $p > n$, la penalización rompe el empate entre las infinitas soluciones perfectas eligiendo una sola: la más regularizada.

**Trade-off clave:** introducimos un poco de **sesgo** (los coeficientes están encogidos hacia cero) a cambio de mucho menos **varianza** (estabilidad, reproducibilidad). En p ≫ n, ese trade es siempre conveniente.

Básicamente, la regularización reduce la sensibilidad del modelo a los datos de entrenamiento, lo que significa que el modelo no se ajustará tan perfectamente a ellos, y a cambio, hará mejores predicciones para nuevos datos.

## Dataset de hoy

Vamos a usar el **breast cancer Wisconsin** (sklearn): 569 muestras, 30 variables medidas a partir de imágenes de núcleos celulares de aspirados por aguja fina. Clase: maligno (212) vs benigno (357).

**Por qué este dataset:** a propósito tiene $n > p$ (569 > 30), así que la regresión clásica **funciona**. Esto nos permite comparar lado a lado clásica vs regularizada, en condiciones donde ambas son aplicables. El dataset también tiene **multicolinealidad alta** entre features geométricamente relacionadas, así que vamos a ver los beneficios de la regularización con claridad.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
nombres = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Escalado: CRÍTICO para regularización
# Si las variables están en escalas distintas, la penalización las castiga
# de forma desigual (las de mayor magnitud reciben más penalización).
# Estandarizar pone a todas en la misma escala (media 0, desvío 1).
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Balance train: {np.bincount(y_train)}")
print(f"Primeras 5 variables: {list(nombres[:5])}")

Train: (398, 30), Test: (171, 30)
Balance train: [148 250]
Primeras 5 variables: [np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness')]


**Detalle importantísimo sobre escalado:** fijate que escalamos **con `fit_transform` en train** y **solo `transform` en test**. Esto NO es un detalle estético — es prevención de **data leakage**:

- `fit_transform(X_train)`: calcula media y desvío usando SOLO el train, y los aplica al train.
- `transform(X_test)`: aplica esa misma media y desvío al test (sin recalcular).

Si hubieras hecho `scaler.fit_transform(X)` sobre todo el dataset antes de partir, el test "habría visto" la media y varianza global. Lo vamos a profundizar en Clase 6.

In [1]:
# Verifiquemos la multicolinealidad: matriz de correlaciones
import pandas as pd
df = pd.DataFrame(X_train_s, columns=nombres)
corr = df.corr().abs()
# Promedio de correlación absoluta (excluyendo diagonal)
n_p = corr.shape[0]
mean_corr = (corr.sum().sum() - n_p) / (n_p * (n_p - 1))
print(f"Correlación absoluta promedio entre variables: {mean_corr:.3f}")

# Cuántos pares tienen correlación > 0.9
pares_altos = ((corr > 0.9).sum().sum() - n_p) // 2
print(f"Pares con |corr| > 0.9: {pares_altos}")

NameError: name 'X_train_s' is not defined

**Interpretación:**

La correlación absoluta promedio entre variables va a ser alrededor de **0.4** — bastante alta para un dataset estadístico genérico. Y vas a encontrar **varios pares con correlación > 0.9** (probablemente alrededor de 15-20 pares).

¿Cuáles? Esencialmente las features que miden lo mismo en escalas distintas:
- "mean radius", "mean perimeter", "mean area" son geometría: si crece el radio, crecen perímetro y área proporcionalmente.
- "worst radius", "worst perimeter", "worst area" son el mismo grupo pero medidos sobre los peores núcleos.
- "mean concave points" y "mean concavity" describen aspectos relacionados de la forma de los núcleos.

Esta estructura — variables medidas redundantemente del mismo fenómeno físico/biológico — es **exactamente lo que regularización resuelve bien y la regresión clásica trata mal**.

---

# Ridge (regularización L2)

## La penalización L2

Ridge minimiza:

$$\hat{\beta}_{\text{Ridge}} = \arg\min_{\beta} \left\{ \|y - X\beta\|^2 + \lambda \sum_{j=1}^p \beta_j^2 \right\}$$

La penalización L2 es **la suma de los cuadrados de los coeficientes**. Es una bola: en 2D, el contorno $\beta_1^2 + \beta_2^2 = $ constante es un círculo; en $p$ dimensiones, una hiperesfera.

> **Convención de sklearn:** el parámetro de regularización se llama `alpha` (α), no λ. Pero matemáticamente son lo mismo. **Más α = más regularización**. Importante: en `LogisticRegression` el parámetro se llama `C` y es el **inverso** de α — más C, MENOS regularización. Esta inconsistencia es histórica y un dolor de cabeza, pero hay que conocerla.

## Por qué siempre tiene solución

Aún cuando $X^T X$ es singular (cuando $p > n$), agregar $\lambda I$ a la matriz lo arregla:

$$\hat{\beta}_{\text{Ridge}} = (X^T X + \lambda I)^{-1} X^T y$$

Sumar $\lambda I$ (con $\lambda > 0$) garantiza que la matriz sea **invertible** porque le agrega $\lambda$ a cada autovalor. Geométricamente, "infla" un poco la matriz en todas las direcciones para que no sea degenerada.

**Esto resuelve el problema 1** del bloque anterior: ahora siempre hay una solución única, incluso con $p > n$.

## Qué le hace Ridge a los coeficientes

Ridge **encoge** los coeficientes hacia cero, pero **nunca exactamente a cero**. Te paso la intuición visual:

Imaginá que en el espacio de coeficientes ($\beta_1$, $\beta_2$), tenés:
- **Las curvas de nivel del error** (elipses concéntricas alrededor del mínimo cuadrático no regularizado).
- **La bola de penalización L2** (un círculo centrado en el origen).

La solución regularizada es el punto donde una curva de nivel del error **toca** la bola de penalización. Como la bola es un **círculo** (suave, sin esquinas), el punto de contacto típicamente tiene **todas** las coordenadas distintas de cero — solo más pequeñas que el mínimo no regularizado.

> Vamos a ver la versión gráfica de esto en el Bloque 3 cuando comparemos con Lasso.

## Lo que Ridge hace bien: multicolinealidad

Cuando dos variables están altamente correlacionadas, Ridge **reparte el coeficiente entre ambas** en lugar de asignarle todo a una.

**Ejemplo intuitivo:** si $x_1$ y $x_2$ son casi idénticas y el verdadero efecto es $y = 3 x_1 + \epsilon$, la regresión clásica podría devolver $\beta_1 = 1500, \beta_2 = -1497$ (porque cualquier combinación que sume 3 sirve). Ridge devuelve algo cercano a $\beta_1 = 1.5, \beta_2 = 1.5$: reparte equitativamente y prefiere coeficientes chicos.

Esto tiene **dos consecuencias importantes en bioinformática:**
- **Buena:** la solución es estable y reproducible.
- **Mala (si querés interpretar):** Ridge no te dice "este gen es el importante", te dice "este grupo de genes correlacionados contribuye en conjunto". Si querés identificar **un** gen como biomarcador, Ridge no sirve.

## Ridge en práctica

Vamos a hacer un ejemplo de regresión: predecir `mean radius` desde el resto de las variables. Es un ejercicio "tonto" biológicamente (sabemos que perimeter y area están correlacionados con radius), pero **exactamente por eso** es ideal para mostrar cómo Ridge maneja la multicolinealidad.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

# Trabajamos con y continuo: predecimos "mean radius" desde el resto de variables
y_reg = X_train_s[:, 0]   # mean radius (estandarizada)
X_reg = X_train_s[:, 1:]  # resto de las variables (29)

y_reg_test = X_test_s[:, 0]
X_reg_test = X_test_s[:, 1:]

# Regresión clásica (sin regularización)
lr = LinearRegression()
lr.fit(X_reg, y_reg)
print(f"Regresión clásica:")
print(f"  R² train: {lr.score(X_reg, y_reg):.4f}")
print(f"  R² test:  {lr.score(X_reg_test, y_reg_test):.4f}")
print(f"  Máx |coef|: {np.abs(lr.coef_).max():.3f}")

# Ridge con varios alphas
print(f"\nRidge:")
for alpha in [0.01, 1.0, 100, 10000]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_reg, y_reg)
    print(f"  α = {alpha:>7.2f}  →  R² test = {ridge.score(X_reg_test, y_reg_test):.4f}, "
          f"Máx |coef| = {np.abs(ridge.coef_).max():.3f}")

Regresión clásica:
  R² train: 0.9997
  R² test:  0.9996
  Máx |coef|: 0.888

Ridge:
  α =    0.01  →  R² test = 0.9996, Máx |coef| = 0.879
  α =    1.00  →  R² test = 0.9987, Máx |coef| = 0.639
  α =  100.00  →  R² test = 0.9709, Máx |coef| = 0.208
  α = 10000.00  →  R² test = 0.4577, Máx |coef| = 0.029


**Cómo leer esta salida:**

- **Regresión clásica:** R² en test suele ser muy alto (~0.999) porque hay variables casi idénticas a `mean radius`. Pero mirá los coeficientes — algunos van a ser grandes, signo de la inestabilidad por multicolinealidad.

- **Ridge con α bajo (0.01):** prácticamente idéntico a la clásica. La regularización es tan débil que no cambia nada.

- **Ridge con α medio (1.0):** R² casi igual, pero coeficientes más chicos. Modelo más estable, igual de bueno predictivamente.

- **Ridge con α alto (100):** R² empieza a caer un poco. La regularización se vuelve demasiado fuerte y el modelo empieza a "sub-ajustar".

- **Ridge con α extremo (10000):** R² cae bastante. Los coeficientes están tan encogidos que el modelo ya no captura la relación.

**Conclusión:** existe un α "óptimo" que es ni muy chico ni muy grande. Lo buscamos por cross-validation, no a ojo. Ese es el tema del ejercicio 1.

## Ejercicio 1 — Ruta de Ridge

**Tu tarea:**

a) Para una grilla de $\alpha \in \{10^{-3}, 10^{-2}, ..., 10^3\}$, ajustá Ridge y guardá para cada valor:
   - Los coeficientes de las **primeras 5 variables**.
   - El R² en test.

b) Hacé dos gráficos lado a lado:
   - **Izquierdo:** los 5 coeficientes en función de $\log(\alpha)$. Cada variable es una línea.
   - **Derecho:** R² test en función de $\log(\alpha)$.

c) Respondé:
   - ¿Cómo se comportan los coeficientes al aumentar $\alpha$? ¿Convergen a algún valor común?
   - ¿Dónde está el $\alpha$ óptimo según el R² test?
   - ¿Coincide con el punto donde los coeficientes "se estabilizan"?

> **Pista:** vas a ver que el R² test forma una **curva en U invertida**. Al principio (α chico) baja un poco por sobreajuste; en el medio está el óptimo; al final (α grande) cae por sub-ajuste. Esto es el clásico **trade-off sesgo-varianza** en acción.

In [ ]:
# TU CÓDIGO ACÁ
alphas = np.logspace(-3, 3, 25)
coefs_path = []
r2_test = []

# COMPLETÁ EL BUCLE: para cada alpha, ajustar Ridge y guardar
# - ridge.coef_[:5] en coefs_path
# - ridge.score(X_reg_test, y_reg_test) en r2_test

# COMPLETÁ LA VISUALIZACIÓN (dos subplots)
# Izquierdo: 5 líneas (una por variable) de coeficiente vs log10(alpha)
# Derecho: R² test vs log10(alpha)

---

# Lasso (regularización L1)

## La penalización L1

Lasso minimiza:

$$\hat{\beta}_{\text{Lasso}} = \arg\min_{\beta} \left\{ \|y - X\beta\|^2 + \lambda \sum_{j=1}^p |\beta_j| \right\}$$

La penalización L1 es la **suma de valores absolutos** de los coeficientes. La diferencia con L2 parece insignificante —"al cuadrado" vs "valor absoluto"— pero las consecuencias son enormes.

## Por qué Lasso selecciona variables (la intuición geométrica)

Acá está la clave. Comparemos las "bolas" de penalización L1 y L2 en 2D:

**L2 (Ridge):** $\beta_1^2 + \beta_2^2 \leq c$ → un **círculo**.
**L1 (Lasso):** $|\beta_1| + |\beta_2| \leq c$ → un **rombo** (cuadrado rotado 45°).

Cuando buscamos el punto donde una curva de nivel del error toca la bola de penalización:

- En el **círculo** (Ridge), el punto de contacto está casi siempre en un punto "lateral" — todas las coordenadas son distintas de cero.

- En el **rombo** (Lasso), el rombo tiene **esquinas** sobre los ejes. La elipse del error tiene una probabilidad alta de tocar una **esquina** del rombo, y eso significa que **una de las coordenadas es exactamente cero**.

Esa propiedad geométrica es la razón por la cual Lasso selecciona variables: las esquinas de la bola L1 son los "puntos preferidos" del optimizador, y en esas esquinas algunos $\beta_j = 0$ exacto.

> Una forma de verlo: L2 es suave, L1 tiene esquinas. Las esquinas del L1 están en los ejes. En los ejes, algunos coeficientes son cero. Cuando el optimizador "se engancha" en una esquina, esos coeficientes se quedan en cero.

## Diferencias entre Lasso y Ridge

| Aspecto | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Selecciona variables | No | **Sí** |
| Maneja multicolinealidad | **Bien** (reparte coeficiente) | Mal (elige una arbitrariamente) |
| Solución cerrada | Sí | No (algoritmo iterativo) |
| Estabilidad numérica | Excelente | Buena |
| Cuándo usar | Esperás que muchas variables aporten algo | Esperás que pocas variables sean clave |

## La limitación principal de Lasso: variables correlacionadas

Acá hay un **problema serio** que conviene tener presente. Si tenés dos variables casi idénticas (ejemplo: `mean radius` y `mean perimeter`), Lasso va a elegir **una sola** y descartar la otra, **de forma esencialmente arbitraria** — depende de detalles numéricos del solver.

**Por qué esto es malo en biología:** en bioinformática, las variables correlacionadas suelen estar correlacionadas **por una razón biológica**: pertenecen al mismo pathway, son targets del mismo factor de transcripción, son genes co-expresados. Tirar 9 de 10 variables de un módulo coherente porque Lasso decidió arbitrariamente quedarse con una sola es **perder información biológica importante**.

Por eso muchas veces **Elastic Net** (próximo bloque) es preferible a Lasso puro: combina la sparsidad de L1 con la "repartición" de L2.

## Lasso en práctica

In [ ]:
from sklearn.linear_model import Lasso

print(f"Lasso (predecir 'mean radius'):")
print(f"{'α':>8} | {'R² test':>9} | {'# vars seleccionadas':>20}")
print("-" * 45)

for alpha in [0.001, 0.01, 0.1, 1.0]:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_reg, y_reg)
    n_nonzero = np.sum(np.abs(lasso.coef_) > 1e-8)
    r2 = lasso.score(X_reg_test, y_reg_test)
    print(f"{alpha:>8.3f} | {r2:>9.4f} | {n_nonzero:>20d}")

**Cómo leer esta salida:**

- **α = 0.001:** muy poca regularización. Probablemente quedan ~25-29 variables con coeficiente no nulo. Casi igual a la regresión clásica.

- **α = 0.01:** ya empieza a hacer selección. Probablemente quedan ~10-15 variables.

- **α = 0.1:** selección agresiva. Probablemente 2-5 variables sobreviven.

- **α = 1.0:** demasiada regularización. Todos los coeficientes son cero — el modelo predice la media. R² puede dar negativo.

Lo importante: **a medida que aumenta α, Lasso va eliminando variables** una por una. El "Lasso path" (que vas a graficar en el ejercicio) muestra esto explícitamente — algunas variables sobreviven hasta α alto, otras se eliminan rápido.

## Ejercicio 2 — Lasso path

**Tu tarea:**

a) Graficá la **ruta del Lasso**: cómo cambian los coeficientes de TODAS las variables al variar $\alpha$. Cada variable es una línea, el eje x es $\log(\alpha)$, el eje y son los coeficientes.

b) Graficá el **número de variables seleccionadas** (coeficientes no nulos) en función de $\log(\alpha)$.

c) Respondé:
   - ¿Las variables se eliminan todas juntas o de a una?
   - ¿Cuáles son las "últimas en irse" (las que sobreviven hasta α más alto)? Biológicamente, ¿qué interpretarías de eso?
   - Compará con Ridge del ejercicio anterior: ¿qué hace Ridge con variables que en Lasso terminan en cero?

> **Pista:** podés usar `sklearn.linear_model.lasso_path()` que devuelve los coeficientes a lo largo de toda una grilla de alphas, o un bucle como en el ejercicio anterior.

In [ ]:
# TU CÓDIGO ACÁ

# a) Lasso path: coeficientes de todas las variables vs alpha
# Opción A (más simple): bucle sobre alphas
# Opción B (más rápido): from sklearn.linear_model import lasso_path

# b) Número de variables no nulas vs alpha

# c) Reflexión en celda Markdown

---

# Elastic Net

## El mejor de los dos mundos

Elastic Net combina las dos penalizaciones:

$$\hat{\beta}_{\text{EN}} = \arg\min_{\beta} \left\{ \|y - X\beta\|^2 + \lambda \left( \rho \sum_j |\beta_j| + (1-\rho) \sum_j \beta_j^2 \right) \right\}$$

Hay **dos parámetros**:
- $\lambda$ (en sklearn: `alpha`): cuánta regularización total. Como antes.
- $\rho$ (en sklearn: `l1_ratio`): qué fracción de la regularización es L1 vs L2.

Casos especiales:
- $\rho = 1$ → Lasso puro.
- $\rho = 0$ → Ridge puro.
- $\rho \in (0, 1)$ → mezcla.

En la práctica $\rho \in \{0.1, 0.5, 0.7, 0.9, 0.99\}$ son valores típicos para probar por CV.

## Por qué Elastic Net resuelve el problema de Lasso con correlaciones

Recordá el problema: si dos variables están correlacionadas, Lasso elige una arbitrariamente.

Elastic Net agrega el término L2, que tiene la propiedad de **repartir coeficientes entre variables correlacionadas**. La combinación:
- El componente L1 mantiene la **sparsidad** (muchos coeficientes en cero exacto).
- El componente L2 evita que la elección entre variables correlacionadas sea arbitraria — **las mantiene juntas** si todas son relevantes.

> Esto se llama el **"grouping effect"** de Elastic Net en la literatura: si dos variables están altamente correlacionadas y son ambas relevantes, Elastic Net les asigna coeficientes similares en lugar de elegir una.

## Cuándo conviene cada uno

La pregunta práctica: ¿cuándo Lasso, cuándo Ridge, cuándo Elastic Net?

**Lasso puro** ($\rho = 1$):
- Esperás **pocas variables relevantes** y muchas irrelevantes.
- Querés **interpretabilidad máxima** (firma chiquita).
- Las variables no son tan correlacionadas entre sí.
- Ejemplo: identificar un puñado de SNPs causales en GWAS.

**Ridge puro** ($\rho = 0$):
- Esperás que **muchas variables aporten algo** (señal distribuida).
- Tenés multicolinealidad fuerte.
- No te importa tener interpretabilidad de pocas variables.
- Ejemplo: predecir una expresión génica desde miles de otros genes co-expresados.

**Elastic Net** ($\rho \in (0, 1)$):
- Querés algo de *sparsity* PERO también que mantenga grupos correlacionados juntos.
- Es el **default seguro** cuando no sabés bien qué esperar.
- Ejemplo: identificar **pathways** (grupos de genes) asociados a un fenotipo.

> **Mi recomendación práctica:** si tenés que elegir uno sin pensar mucho, probá Elastic Net con `l1_ratio=0.5` y CV. Suele dar buenos resultados en una amplia gama de problemas.

## Elastic Net en práctica

In [ ]:
from sklearn.linear_model import ElasticNet

print(f"Elastic Net (α = 0.05 fijo, variando l1_ratio):")
print(f"{'l1_ratio':>10} | {'Interpretación':>20} | {'R² test':>9} | {'# vars':>7}")
print("-" * 55)

for l1_ratio, descripcion in [
    (0.1, "Casi Ridge"),
    (0.5, "Mezcla 50/50"),
    (0.9, "Más bien Lasso"),
    (1.0, "Lasso puro")
]:
    en = ElasticNet(alpha=0.05, l1_ratio=l1_ratio, max_iter=10000)
    en.fit(X_reg, y_reg)
    n_nonzero = np.sum(np.abs(en.coef_) > 1e-8)
    r2 = en.score(X_reg_test, y_reg_test)
    print(f"{l1_ratio:>10.1f} | {descripcion:>20} | {r2:>9.4f} | {n_nonzero:>7d}")

Elastic Net (α = 0.05 fijo, variando l1_ratio):
  l1_ratio |       Interpretación |   R² test |  # vars
-------------------------------------------------------
       0.1 |           Casi Ridge |    0.9905 |      16
       0.5 |         Mezcla 50/50 |    0.9927 |       5
       0.9 |       Más bien Lasso |    0.9946 |       4
       1.0 |           Lasso puro |    0.9950 |       3


**Cómo leer esta salida:**

A medida que `l1_ratio` aumenta de 0 a 1:
- El **número de variables seleccionadas baja** (más cerca de Lasso = más sparsidad).
- El **R² puede mantenerse o subir** levemente.

Lo importante: con `l1_ratio = 0.5` (mezcla), ya tenés selección bastante agresiva pero más estable que Lasso puro, porque mantiene grupos correlacionados juntos.

## Ejercicio 3 — Comparar Ridge, Lasso, Elastic Net por CV

**Tu tarea:**

Para los tres métodos, encontrá el $\alpha$ óptimo por **5-fold cross-validation** sobre el train. Después, evaluá en test y completá la tabla:

| Método | α óptimo | l1_ratio óptimo | R² test | # variables seleccionadas |
|---|---|---|---|---|
| Ridge | | (no aplica) | | 29 (no selecciona) |
| Lasso | | (no aplica) | | |
| Elastic Net | | | | |

Sklearn provee versiones con CV automático que hacen todo el trabajo:
- `RidgeCV(alphas=grilla)`: prueba la grilla y se queda con el mejor.
- `LassoCV(alphas=grilla, cv=5)`: similar pero para Lasso.
- `ElasticNetCV(alphas=grilla, l1_ratio=[lista_de_ratios], cv=5)`: ajusta α **y** l1_ratio simultáneamente.

Después de ajustar, podés mirar `.alpha_`, `.l1_ratio_` y `.coef_` para inspeccionar el modelo elegido.

**Pregunta final:** ¿hay algún método que claramente gane en este problema? ¿O dan los tres resultados similares en performance pero distintos en sparsidad?

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# TU CÓDIGO ACÁ
# 1. RidgeCV con alphas = np.logspace(-3, 3, 50)
# 2. LassoCV con alphas = np.logspace(-4, 0, 50), cv=5
# 3. ElasticNetCV con alphas = np.logspace(-4, 0, 50),
#                  l1_ratio = [0.1, 0.5, 0.7, 0.9, 0.99], cv=5

# Para cada uno:
#   - Mostrar alpha óptimo (y l1_ratio para EN)
#   - Mostrar R² en test
#   - Contar variables con coeficiente no nulo

---

# Regresión logística regularizada (clasificación)

## Del problema de regresión al de clasificación

Hasta ahora vimos regresión  para predecir un valor **continuo** ($y \in \mathbb{R}$). Ahora **clasificación binaria**: predecir una **clase** ($y \in \{0, 1\}$).

La regresión logística modela la probabilidad de la clase 1 como:

$$P(y=1 \mid x) = \frac{1}{1 + e^{-x^T \beta}} = \sigma(x^T \beta)$$

donde $\sigma$ es la función sigmoide. El modelo se ajusta maximizando la **log-verosimilitud** (equivalente a minimizar la log-loss / cross-entropy).

## La regularización funciona igual

Igual que en regresión, podemos agregar penalizaciones a la log-verosimilitud:

$$\hat{\beta} = \arg\max_{\beta} \left\{ \log L(\beta) - \lambda \cdot \text{penalización}(\beta) \right\}$$

Las tres opciones siguen siendo L1, L2, L1+L2. **La intuición geométrica y las propiedades son las mismas:** Lasso selecciona, Ridge encoge sin seleccionar, Elastic Net combina.

## La trampa de sklearn: C vs alpha

Acá hay un **detalle de implementación que confunde a todo el mundo**. En `LogisticRegression`:

$$\hat{\beta} = \arg\max_{\beta} \left\{ C \cdot \log L(\beta) - \text{penalización}(\beta) \right\}$$

El parámetro es **`C` y es el INVERSO de la regularización**. Más C = MENOS regularización.

| Convención | Parámetro | Significado |
|---|---|---|
| `Lasso`, `Ridge`, `ElasticNet` | `alpha` (α) | **Más α = MÁS regularización** |
| `LogisticRegression`, `SVC` | `C` | **Más C = MENOS regularización** |

Equivalencia: $C \approx 1/\alpha$.

**Por qué esto importa:** si copiás código de un ejercicio de Ridge a uno de logística sin cambiar la dirección, vas a tener mil dolores de cabeza. Cuando uses `LogisticRegression` con regularización, **acordate de invertir la lógica**: para regularización fuerte usá `C` chico (ej: 0.01), para débil usá `C` grande (ej: 100).

## Solver: cuál usar para cada penalización

Sklearn usa distintos algoritmos según la penalización:

| Penalización | Solver recomendado |
|---|---|
| L2 (default) | `'lbfgs'`, `'liblinear'` |
| L1 | `'liblinear'`, `'saga'` |
| Elastic Net | `'saga'` (único que lo soporta) |
| Sin regularización | cualquiera, con `penalty=None` |

Si pedís L1 y no especificás solver, sklearn tira un error. Si pedís Elastic Net sin `saga`, también. Mejor especificar siempre.

## Logística regularizada en práctica

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print(f"Clasificación de tumores: maligno vs benigno (breast cancer Wisconsin)")
print(f"{'Método':>22} | {'C':>8} | {'Acc test':>9} | {'AUC':>5} | {'# vars':>7}")
print("-" * 65)

for nombre, penalty, C, l1_ratio in [
    ("Sin regularización",   None,         1e10, None),
    ("L2 (Ridge, C=1)",      "l2",         1.0,  None),
    ("L2 (Ridge, C=0.01)",   "l2",         0.01, None),
    ("L1 (Lasso, C=0.1)",    "l1",         0.1,  None),
    ("L1 (Lasso, C=0.01)",   "l1",         0.01, None),
    ("Elastic Net (C=0.1)",  "elasticnet", 0.1,  0.5),
]:
    if penalty == "elasticnet":
        clf = LogisticRegression(penalty="elasticnet", solver="saga",
                                  l1_ratio=l1_ratio, C=C, max_iter=10000)
    elif penalty is None:
        clf = LogisticRegression(penalty=None, max_iter=10000)
    else:
        clf = LogisticRegression(penalty=penalty, solver="liblinear", C=C, max_iter=10000)

    clf.fit(X_train_s, y_train)
    pred = clf.predict(X_test_s)
    proba = clf.predict_proba(X_test_s)[:, 1]
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    n_vars = np.sum(np.abs(clf.coef_[0]) > 1e-8)
    print(f"{nombre:>22} | {C:>8.2f} | {acc:>9.3f} | {auc:>5.3f} | {n_vars:>7d}")

**Cómo leer esta tabla:**

- **Sin regularización:** usa todas las 30 variables. Performance buena pero potencialmente sobreajustada.

- **L2 con C=1:** regularización moderada, sigue usando las 30 variables (Ridge no selecciona), pero coeficientes encogidos. AUC suele ser **igual o ligeramente mejor** que sin regularización (la regularización ayuda incluso cuando n > p, por la multicolinealidad).

- **L2 con C=0.01:** mucha regularización, performance puede empezar a caer.

- **L1 con C=0.1:** selecciona ~7-10 variables. AUC casi igual a usar las 30. Esto es genial: **modelo mucho más simple e interpretable con misma performance**.

- **L1 con C=0.01:** demasiada regularización, solo sobreviven 2-3 variables, performance cae.

- **Elastic Net con C=0.1, l1_ratio=0.5:** intermedio entre L2 y L1. Selecciona más variables que Lasso puro pero menos que Ridge.

**Moraleja:** en este dataset, **Ridge con C=1** es probablemente la mejor opción si querés máxima performance. **Lasso con C=0.1** es la mejor si querés interpretabilidad (firma chica) sin perder mucho.

## Ejercicio 4 — Logística regularizada y biología

**Tu tarea:**

a) Ajustá una `LogisticRegression` con L1 y `C` elegido por **CV 5-fold**. La forma más fácil: usar `LogisticRegressionCV`, que hace lo mismo que `LassoCV` pero para clasificación.

```python
from sklearn.linear_model import LogisticRegressionCV
clf = LogisticRegressionCV(
    Cs=np.logspace(-3, 1, 20),
    penalty="l1", solver="liblinear",
    cv=5, max_iter=10000, random_state=0
)
```

¿Cuál es el `C` óptimo? ¿Cuántas variables sobreviven?

b) Mostrá los nombres de las variables seleccionadas y sus coeficientes ordenados por magnitud (valor absoluto). ¿Qué interpretás biológicamente?

c) **Pregunta de validación:** ¿podemos confiar en que estas variables son "biomarcadores"? ¿Qué pasos adicionales harías antes de publicar esto como una "firma diagnóstica"?

> **Pista para c):** pensá en lo que vimos en Clase 2 (estabilidad por bootstrap), lo que viene en Clase 6 (nested CV, validación externa). Una buena respuesta menciona al menos 3 cosas: estabilidad de la selección, validación externa, plausibilidad biológica.

---

# Cierre — Ejercicio integrador

## Ejercicio 5 — Decisión metodológica (escrito)

Te pasan tres situaciones de bioinformática. Para cada una, indicá:
- Qué método de regularización elegirías (Ridge, Lasso, Elastic Net).
- Por qué (en una o dos frases).
- Qué cuidados adicionales tomarías.

**Situación A — GWAS de respuesta a tratamiento:**
Querés predecir respuesta a tratamiento (binaria) desde **30.000 SNPs** medidos en **200 pacientes**. Tu hipótesis biológica es que **pocas variantes genéticas** tienen efectos grandes y específicos, y el resto son ruido. Querés terminar con una lista publicable de SNPs candidatos.

**Situación B — Predicción de expresión génica:**
Querés modelar la expresión de un gen target (continuo) desde **el resto del transcriptoma** (20.000 genes) en una cohorte de 5.000 muestras. Tu hipótesis biológica es que **muchísimos genes** contribuyen en pequeña medida a la regulación del target, y que hay **módulos de genes co-regulados** que actúan en conjunto.

**Situación C — Clasificación de imágenes histopatológicas:**
Tenés **500 features** extraídas de imágenes de tumores por un pipeline automatizado, medidas en **1.000 muestras** ($n > p$). Las features están muy correlacionadas porque salen del mismo procesamiento (medidas de textura en distintas escalas, momentos estadísticos, etc.). Querés un clasificador que prediga grado tumoral.

**Tus respuestas:**

- **A:** método ___ porque ___. Cuidados: ___
- **B:** método ___ porque ___. Cuidados: ___
- **C:** método ___ porque ___. Cuidados: ___

---

## Resumen de la clase

| Método | Penalización | Selecciona variables? | Trata multicolinealidad? | Cuándo usar |
|---|---|---|---|---|
| **Ridge** | L2 (suma de cuadrados) | No (encoge sin eliminar) | **Bien** (reparte) | Muchas variables relevantes, multicolinealidad fuerte |
| **Lasso** | L1 (suma de absolutos) | **Sí** (sparse) | Mal (elige una arbitraria) | Pocas variables relevantes |
| **Elastic Net** | L1 + L2 | Sí | Bien | Combina lo mejor de los dos. Default seguro. |

## Reglas prácticas para llevarse

1. **Siempre escalar las variables** antes de aplicar regularización. Sin escalado, la penalización es injusta para variables con escalas distintas.

2. **Siempre elegir $\alpha$/C por cross-validation**, nunca a ojo. Usá `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegressionCV`.

3. **Cuidado con la dirección del parámetro:**
   - `Lasso`/`Ridge`/`ElasticNet`: **más α = más regularización**.
   - `LogisticRegression`/`SVC`: **más C = MENOS regularización**.
   - Equivalencia: $C \approx 1/\alpha$.

4. **Para interpretabilidad biológica**: combiná Lasso con el **bootstrap de estabilidad** de Clase 2. Una "firma" que aparece en el 70%+ de los bootstraps es defendible; una que aparece solo una vez es ruido.

5. **Pipeline para evitar data leakage:** todo lo que aprende parámetros del train (escalado, selección, ajuste del modelo) debe estar en un `Pipeline` que se aplica dentro del CV. Profundizamos en Clase 6.

6. **Si no sabés por dónde empezar:** Elastic Net con `l1_ratio=0.5` y CV. Es un buen default que funciona en muchos escenarios sin tener que pensar demasiado.

## Conexión con la próxima clase

Hoy vimos **modelos lineales** con regularización. Funcionan muy bien cuando las clases (o las relaciones entre $X$ e $y$) son **aproximadamente lineales**. Pero a veces no lo son.

En la **Clase 5** vamos a ver **Support Vector Machines (SVM)** y el **truco del kernel**, que nos permite hacer clasificación **no lineal** sin sufrir las complicaciones de calcular el mapeo a un espacio de alta dimensión.

Y vas a ver una conexión profunda: **el parámetro C de SVM es exactamente regularización L2**, igual que Ridge. La diferencia entre SVM y Ridge no es la regularización — es la función de pérdida (hinge loss vs squared loss). Pero la idea de regularización es la misma.

Eso ilustra algo importante del curso: muchas técnicas de ML aparentemente distintas comparten el mismo esqueleto matemático. **Una vez que entendés regularización, entendés la mitad del machine learning moderno.**

---

*Bibliografía y videos en `bibliografia_clase4.md`.*